# 00 - Project Setup

Run this once before the pipeline. It:
1. Creates the Unity Catalog catalog and schemas
2. Creates the `source_files` volume for the source CSV files
3. Checks that both CSV files were uploaded to the volume

Safe to re-run: every statement uses `IF NOT EXISTS`.

## 1. Configuration

In [0]:
from configs.etl_config import (
    CROP_FILE_NAME,
    WEATHER_FILE_NAME,
    CATALOG,
    SCHEMAS_TO_CREATE,
    SOURCE_VOLUME_SCHEMA,
    SOURCE_VOLUME_NAME,
    SOURCE_VOLUME_PATH,
)

print(f"Catalog     : {CATALOG}")
print(f"Volume path : {SOURCE_VOLUME_PATH}")

## 2. Create catalog, schemas and volume

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")

for schema_name in SCHEMAS_TO_CREATE:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema_name}")
    print(f"Schema ready: {CATALOG}.{schema_name}")

spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SOURCE_VOLUME_SCHEMA}.{SOURCE_VOLUME_NAME}")
print(f"Volume ready: {SOURCE_VOLUME_PATH}")

## 3. Check the source files

Upload `crop_trend.csv` and `province_weather_quarterly.csv` to the volume
(Catalog Explorer -> `landing` -> `source_files` -> Upload), then re-run this cell.

In [0]:
files_in_volume = {file_info.name for file_info in dbutils.fs.ls(SOURCE_VOLUME_PATH)}

source_file_names = [CROP_FILE_NAME, WEATHER_FILE_NAME]

for file_name in source_file_names:
    status = "[x]" if file_name in files_in_volume else "[ ]"
    print(f"{status} {file_name}")

missing_files = [name for name in source_file_names if name not in files_in_volume]
if missing_files:
    raise FileNotFoundError(f"Upload these files to {SOURCE_VOLUME_PATH}, then re-run: {missing_files}")

print("Setup complete. Next: 01_bronze/01_ingest_source_data")